In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset, Subset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    torch.set_deterministic_debug_mode(True)



################################
#     RESTART     RUNTIME      #
################################
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.train.train import *

import torchvision.models as models

In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
BATCH_SIZE = 256

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse',
           'ship', 'truck')

from torch.utils.data import random_split

full_train_dataset = datasets.CIFAR10(root='./data', train=True,
                                      download=True, transform=transform_train)

test_dataset = datasets.CIFAR10(root='./data', train=False,
                                      download=True, transform=transform_test)

train_dataset, val_dataset = random_split(full_train_dataset, [0.8, 0.2])

# Новая логика: finder — доля train для расширения (100% = полный train)
finder_fraction = 1.0
n_finder = int(finder_fraction * len(train_dataset))
finder_dataset = Subset(train_dataset.dataset, train_dataset.indices[:n_finder])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
finder_loader = DataLoader(finder_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [4]:
class ExpandingHead(nn.Module):
    def __init__(self, num_classes=10):
        super(ExpandingHead, self).__init__()
        
        backbone = models.vgg19_bn(weights=None)
        self.backbone = backbone.features
        self.fc0 = nn.Linear(512, num_classes)
    
    def forward(self, x):
        x = self.backbone(x)
        x = torch.flatten(x, 1)
        x = self.fc0(x)
        return x
    
    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

In [5]:
model = ExpandingHead()

In [ ]:
model( torch.randn(2, 3, 32, 32))

In [ ]:
model = convert_to_scaled_net(model, layers_to_convert=[model.fc0], device=device)

In [8]:
hyperparams = {
    "num_epochs": 256,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc0": 0.01},
    "pruning_thresholds": {"fc0": 1},
    "plateau_threshold": 0.01,
    "min_epochs_between_expansions": 16,
    "plateau_window_size": 5,
    "learning_rate": 1e-3,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 4000,
    "weight_decay": 4e-3,
    "diff_threshold": 0.05
}

In [ ]:
from comet_ml import start
from comet_ml.integration.pytorch import log_model
experiment = start(
    api_key=os.environ.get("COMET_API_KEY"),
    workspace=os.environ.get("COMET_WORKSPACE"),
    project_name="cifar10-backbone",
)
experiment.log_parameters(hyperparams)
experiment.set_name("trash")

In [ ]:
criterion = nn.CrossEntropyLoss()

def optimizer_factory(model_instance):
    return optim.SGD(
        model_instance.parameters(), 
        lr=hyperparams['learning_rate'], 
        weight_decay=hyperparams['weight_decay']
    )

trained_model = train_ssonn(
    model, 
    train_loader, 
    val_loader, 
    finder_loader,
    criterion, 
    optimizer_factory, 
    hyperparams, 
    device,
    experiment,
    True
)

In [ ]:
experiment.end()